# Analisis Awal Penjualan — April s/d September 2020

Sumber data: `Data Penjualan April - Sep 2020.xlsx` (Sheet1 / Table1, 6.779 baris transaksi).

**Skema kolom**

| Kolom | Arti | Tipe |
|---|---|---|
| `TGL` | Tanggal transaksi | tanggal (serial Excel) |
| `KODEBARA` | Kode barang | teks |
| `NAMABARA` | Nama barang | teks |
| `QTY` | Kuantitas | numerik |
| `SATUAN` | Satuan jual (DUS, PACK, dsb.) | teks |
| `HARGA` | Harga per satuan | numerik (Rp) |
| `JUMLAH` | Nilai baris = QTY x HARGA | numerik (Rp) |
| `NAMA` | Nama pelanggan / outlet | teks |

**Isi notebook**
1. Muat data & profil struktur
2. Pemeriksaan kualitas data
3. Pembersihan & kolom turunan
4. KPI ringkas
5. Grafik: tren waktu, produk, pelanggan, brand, satuan, harga
6. Temuan awal

## 1. Muat data & profil struktur

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

FILE = 'Data Penjualan April - Sep 2020.xlsx'

def baca_xlsx_stdlib(path, sheet='xl/worksheets/sheet1.xml'):
    """Cadangan bila openpyxl belum terpasang: baca xlsx (zip+xml) ke DataFrame."""
    import zipfile
    from xml.etree import ElementTree as ET

    ns = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
    z = zipfile.ZipFile(path)
    shared = [''.join(t.text or '' for t in si.iter(ns + 't'))
              for si in ET.fromstring(z.read('xl/sharedStrings.xml'))]
try:
    df = pd.read_excel(FILE, sheet_name='Sheet1')
    print('Dibaca via pd.read_excel (openpyxl).')
except ImportError:
    df = baca_xlsx_stdlib(FILE)
    print('openpyxl tidak tersedia -> memakai pembaca cadangan (zip+xml).')
    print('Agar lebih ringkas ke depan: pip install openpyxl')

print('Dimensi:', df.shape)
df.head()

In [ ]:
# Profil struktur: tipe, nilai kosong, kardinalitas
profil = pd.DataFrame({
    'tipe': df.dtypes.astype(str),
    'kosong': df.isna().sum(),
    'kosong_%': (df.isna().mean() * 100).round(2),
    'nilai_unik': df.nunique(),
    'contoh': [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
profil

## 2. Pembersihan & kolom turunan

- `TGL` disimpan sebagai *serial* Excel (mis. `43922`) bila dibaca lewat jalur cadangan → dikonversi dengan origin `1899-12-30`.
- `NAMA` memakai pola prefiks kanal, mis. `F - (MITRAKU) TOKO SALMAINI` → dipisah menjadi `KANAL`, `GRUP`, dan `PELANGGAN`.
- Brand diambil dari kata pertama `NAMABARA` (JOFRANS, OKEY, BELFOODS, ...).

In [ ]:
d = df.copy()
d.columns = [c.strip().upper() for c in d.columns]

# --- Tanggal ---
if not pd.api.types.is_datetime64_any_dtype(d['TGL']):
    d['TGL'] = pd.to_datetime(d['TGL'].astype(float), unit='D', origin='1899-12-30')

d['BULAN'] = d['TGL'].dt.to_period('M').dt.to_timestamp()
d['HARI'] = d['TGL'].dt.day_name()

# --- Numerik ---
for c in ['QTY', 'HARGA', 'JUMLAH']:
    d[c] = pd.to_numeric(d[c], errors='coerce')

# --- Teks ---
for c in ['KODEBARA', 'NAMABARA', 'SATUAN', 'NAMA']:
    d[c] = d[c].astype('string').str.strip()

# --- Pecah NAMA: "F - (MITRAKU) TOKO SALMAINI HABABAHAN" ---
pola = r'^\s*(?:(?P<KANAL>[A-Z0-9]{1,3})\s*-\s*)?(?:\((?P<GRUP>[^)]*)\)\s*)?(?P<PELANGGAN>.*)$'
pecah = d['NAMA'].str.extract(pola)
d['KANAL'] = pecah['KANAL'].fillna('(tanpa kode)')
d['GRUP'] = pecah['GRUP'].fillna('(tanpa grup)')
d['PELANGGAN'] = pecah['PELANGGAN'].str.strip().replace('', pd.NA).fillna(d['NAMA'])

# --- Nama barang kanonik ---
# 25 kode barang punya lebih dari satu ejaan nama (lihat bagian 3). Tanpa
# penyeragaman ini satu SKU terhitung sebagai beberapa SKU dan peringkat produk
# maupun kurva Pareto ikut melenceng.
kanonik = (d.dropna(subset=['KODEBARA'])
             .groupby('KODEBARA')['NAMABARA']
             .agg(lambda s: s.value_counts().idxmax()))
d['NAMA_KANONIK'] = d['KODEBARA'].map(kanonik).fillna(d['NAMABARA'])

# --- Brand dari nama barang ---
d['BRAND'] = d['NAMA_KANONIK'].str.split().str[0].str.upper()

# --- Konsistensi nilai baris ---
d['JUMLAH_HITUNG'] = d['QTY'] * d['HARGA']
d['SELISIH'] = (d['JUMLAH'] - d['JUMLAH_HITUNG']).round(2)

print('Rentang tanggal :', d['TGL'].min().date(), '->', d['TGL'].max().date())
print('Jumlah baris    :', len(d))
print('SKU unik (kode) :', d['KODEBARA'].nunique(), '| variasi nama mentah:', d['NAMABARA'].nunique())
d[['TGL', 'KODEBARA', 'NAMA_KANONIK', 'QTY', 'SATUAN', 'HARGA', 'JUMLAH', 'KANAL', 'GRUP', 'PELANGGAN', 'BRAND']].head()

## 3. Pemeriksaan kualitas data

Dilakukan **sebelum** analisis agar angka KPI tidak menyesatkan.

In [ ]:
cek = {
    'baris duplikat penuh': int(d.duplicated().sum()),
    'QTY <= 0': int((d['QTY'] <= 0).sum()),
    'HARGA <= 0': int((d['HARGA'] <= 0).sum()),
    'JUMLAH <= 0 (retur?)': int((d['JUMLAH'] <= 0).sum()),
    'JUMLAH != QTY x HARGA': int((d['SELISIH'].abs() > 0.5).sum()),
    'TGL kosong': int(d['TGL'].isna().sum()),
    'NAMA kosong': int(d['NAMA'].isna().sum()),
    'KODEBARA kosong': int(d['KODEBARA'].isna().sum()),
}
ringkas = pd.Series(cek, name='jumlah_baris').to_frame()
ringkas['persen'] = (ringkas['jumlah_baris'] / len(d) * 100).round(2)
display(ringkas)

# Satu kode barang seharusnya satu nama barang — cek penamaan tidak konsisten
nama_ganda = d.groupby('KODEBARA')['NAMABARA'].nunique()
nama_ganda = nama_ganda[nama_ganda > 1]
print(f'\nKode barang dengan >1 variasi nama: {len(nama_ganda)}')
if len(nama_ganda):
    display(d[d['KODEBARA'].isin(nama_ganda.index)]
            .groupby(['KODEBARA', 'NAMABARA']).size().rename('baris').head(20))

In [ ]:
# Contoh baris bermasalah (bila ada) untuk ditelaah manual
masalah = d[(d['SELISIH'].abs() > 0.5) | (d['QTY'] <= 0) | (d['JUMLAH'] <= 0)]
print('Total baris bermasalah:', len(masalah))
masalah[['TGL', 'KODEBARA', 'NAMABARA', 'QTY', 'HARGA', 'JUMLAH', 'JUMLAH_HITUNG', 'SELISIH', 'NAMA']].head(15)

## 4. KPI ringkas

In [ ]:
hari_aktif = d['TGL'].nunique()
kpi = pd.Series({
    'Total omzet (Rp)': d['JUMLAH'].sum(),
    'Total baris transaksi': len(d),
    'Hari dengan penjualan': hari_aktif,
    'Rata-rata omzet / hari aktif (Rp)': d['JUMLAH'].sum() / hari_aktif,
    'Rata-rata nilai per baris (Rp)': d['JUMLAH'].mean(),
    'Median nilai per baris (Rp)': d['JUMLAH'].median(),
    'Jumlah pelanggan unik': d['PELANGGAN'].nunique(),
    'Jumlah SKU unik': d['KODEBARA'].nunique(),
    'Jumlah brand unik': d['BRAND'].nunique(),
    'Total kuantitas terjual': d['QTY'].sum(),
})
kpi.to_frame('nilai')

## 5. Grafik

### 5.1 Tren waktu

In [ ]:
bulanan = d.groupby('BULAN').agg(
    omzet=('JUMLAH', 'sum'),
    baris=('JUMLAH', 'size'),
    qty=('QTY', 'sum'),
    pelanggan=('PELANGGAN', 'nunique'),
)
bulanan['pertumbuhan_%'] = (bulanan['omzet'].pct_change() * 100).round(1)
display(bulanan)

fig, ax = plt.subplots(figsize=(11, 5))
label = bulanan.index.strftime('%b %Y')
bar = ax.bar(label, bulanan['omzet'] / 1e6, color='#4C72B0')
ax.set_title('Omzet per Bulan (April - September 2020)')
ax.set_ylabel('Omzet (juta Rp)')
ax.bar_label(bar, fmt='%.0f', padding=3, fontsize=9)

ax2 = ax.twinx()
ax2.plot(label, bulanan['pelanggan'], color='#C44E52', marker='o', label='Pelanggan aktif')
ax2.set_ylabel('Pelanggan aktif')
ax2.grid(False)
ax2.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
harian = d.groupby('TGL')['JUMLAH'].sum().sort_index()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(harian.index, harian.values / 1e6, lw=0.9, color='#8C8C8C', label='Omzet harian')
ax.plot(harian.index, harian.rolling(7).mean() / 1e6, lw=2.2, color='#4C72B0', label='Rata-rata bergerak 7 hari')
ax.set_title('Omzet Harian dan Rata-rata Bergerak 7 Hari')
ax.set_ylabel('Omzet (juta Rp)')
ax.legend()
plt.tight_layout()
plt.show()

print('5 hari omzet tertinggi:')
display((harian.sort_values(ascending=False).head() / 1e6).round(2).rename('juta Rp'))

In [ ]:
urut = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
per_hari = d.groupby('HARI')['JUMLAH'].sum().reindex(urut).fillna(0)
# dibagi jumlah kemunculan hari agar adil
bobot = d.drop_duplicates('TGL')['HARI'].value_counts().reindex(urut).fillna(0)
rata_hari = (per_hari / bobot.replace(0, np.nan)).fillna(0)

fig, ax = plt.subplots(figsize=(10, 4.5))
b = ax.bar(urut, rata_hari / 1e6, color='#55A868')
ax.bar_label(b, fmt='%.1f', padding=3, fontsize=9)
ax.set_title('Rata-rata Omzet per Hari dalam Seminggu')
ax.set_ylabel('Rata-rata omzet (juta Rp)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### 5.2 Produk

In [ ]:
produk = (d.groupby(['KODEBARA', 'NAMA_KANONIK'])
            .agg(omzet=('JUMLAH', 'sum'), qty=('QTY', 'sum'), transaksi=('JUMLAH', 'size'))
            .sort_values('omzet', ascending=False))
produk['kontribusi_%'] = (produk['omzet'] / produk['omzet'].sum() * 100).round(2)
display(produk.head(20))

top = produk.head(15).reset_index()
nama_pendek = top['NAMA_KANONIK'].str.slice(0, 42) + ' (' + top['KODEBARA'] + ')'

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(nama_pendek[::-1], top['omzet'][::-1] / 1e6, color='#4C72B0')
ax.set_title('15 Produk dengan Omzet Tertinggi')
ax.set_xlabel('Omzet (juta Rp)')
plt.tight_layout()
plt.show()

In [ ]:
# Pareto: berapa sedikit SKU yang menyumbang 80% omzet?
kum = produk['omzet'].cumsum() / produk['omzet'].sum() * 100
n80 = int((kum <= 80).sum() + 1)
print(f'{n80} dari {len(produk)} SKU ({n80/len(produk)*100:.1f}%) menyumbang 80% omzet.')

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(range(1, len(kum) + 1), kum.values, color='#4C72B0', lw=2)
ax.axhline(80, color='#C44E52', ls='--', lw=1)
ax.axvline(n80, color='#C44E52', ls='--', lw=1)
ax.annotate(f'{n80} SKU = 80% omzet', xy=(n80, 80), xytext=(n80 + len(kum) * 0.06, 55),
            arrowprops=dict(arrowstyle='->', color='#C44E52'), color='#C44E52')
ax.set_title('Kurva Pareto Kontribusi Omzet per SKU')
ax.set_xlabel('Peringkat SKU')
ax.set_ylabel('Kontribusi kumulatif (%)')
plt.tight_layout()
plt.show()

### 5.3 Pelanggan & kanal

In [ ]:
pelanggan = (d.groupby('PELANGGAN')
               .agg(omzet=('JUMLAH', 'sum'), transaksi=('JUMLAH', 'size'),
                    hari_beli=('TGL', 'nunique'), sku=('KODEBARA', 'nunique'))
               .sort_values('omzet', ascending=False))
pelanggan['kontribusi_%'] = (pelanggan['omzet'] / pelanggan['omzet'].sum() * 100).round(2)
display(pelanggan.head(20))

topc = pelanggan.head(15)
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(topc.index.str.slice(0, 45)[::-1], topc['omzet'][::-1] / 1e6, color='#DD8452')
ax.set_title('15 Pelanggan dengan Omzet Tertinggi')
ax.set_xlabel('Omzet (juta Rp)')
plt.tight_layout()
plt.show()

kum_c = pelanggan['omzet'].cumsum() / pelanggan['omzet'].sum() * 100
n80c = int((kum_c <= 80).sum() + 1)
print(f'{n80c} dari {len(pelanggan)} pelanggan ({n80c/len(pelanggan)*100:.1f}%) menyumbang 80% omzet.')

In [ ]:
kanal = d.groupby('KANAL')['JUMLAH'].sum().sort_values(ascending=False)
grup = d.groupby('GRUP')['JUMLAH'].sum().sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(kanal.index.astype(str), kanal.values / 1e6, color='#4C72B0')
axes[0].set_title('Omzet per Kode Kanal (prefiks NAMA)')
axes[0].set_ylabel('Omzet (juta Rp)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].barh(grup.index.astype(str)[::-1], grup.values[::-1] / 1e6, color='#937860')
axes[1].set_title('10 Grup Pelanggan Teratas')
axes[1].set_xlabel('Omzet (juta Rp)')
plt.tight_layout()
plt.show()

### 5.4 Brand, satuan, dan harga

In [ ]:
brand = d.groupby('BRAND')['JUMLAH'].sum().sort_values(ascending=False)
satuan = d.groupby('SATUAN').agg(omzet=('JUMLAH', 'sum'), baris=('JUMLAH', 'size'))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
b10 = brand.head(10)
axes[0].barh(b10.index.astype(str)[::-1], b10.values[::-1] / 1e6, color='#55A868')
axes[0].set_title('10 Brand dengan Omzet Tertinggi')
axes[0].set_xlabel('Omzet (juta Rp)')

axes[1].pie(satuan['omzet'], labels=satuan.index.astype(str), autopct='%1.1f%%', startangle=90,
            colors=plt.cm.Set2.colors)
axes[1].set_title('Komposisi Omzet per Satuan Jual')
plt.tight_layout()
plt.show()

display(satuan.sort_values('omzet', ascending=False))

In [ ]:
# Sebaran nilai baris + konsistensi harga.
# Catatan: harga HARUS dibandingkan per (KODEBARA, SATUAN). Satu kode yang dijual
# dalam DUS dan PACK wajar punya harga berbeda; yang perlu ditelaah adalah harga
# berbeda untuk kode DAN satuan yang sama.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(d['JUMLAH'] / 1e6, bins=60, color='#4C72B0')
axes[0].set_yscale('log')
axes[0].set_title('Sebaran Nilai per Baris Transaksi')
axes[0].set_xlabel('Nilai baris (juta Rp)')
axes[0].set_ylabel('Frekuensi (skala log)')

var_harga = (d[d['HARGA'] > 0]
             .groupby(['KODEBARA', 'SATUAN'])['HARGA']
             .agg(['min', 'max', 'nunique', 'mean', 'size'])
             .rename(columns={'size': 'baris'}))
var_harga['rentang_pct'] = ((var_harga['max'] - var_harga['min']) / var_harga['mean'] * 100).round(1)

# hanya kombinasi yang cukup sering muncul, agar bukan sekadar kasus tunggal
vh = (var_harga[(var_harga['nunique'] > 1) & (var_harga['baris'] >= 5)]
      .sort_values('rentang_pct', ascending=False).head(12))
label_vh = [f'{k} / {s}' for k, s in vh.index]

axes[1].barh(label_vh[::-1], vh['rentang_pct'][::-1], color='#C44E52')
axes[1].set_title('Harga Paling Tidak Konsisten (kode + satuan sama)')
axes[1].set_xlabel('(max - min) / rata-rata  (%)')
plt.tight_layout()
plt.show()

n_var = int((var_harga['nunique'] > 1).sum())
print(f'Kombinasi kode+satuan dengan >1 tingkat harga: {n_var} dari {len(var_harga)} '
      f'({n_var / len(var_harga) * 100:.1f}%)')
print('Sebagian wajar (diskon/tier pelanggan), sebagian lain layak dicek sebagai salah input.')
display(vh)

In [ ]:
# Heatmap: omzet brand teratas per bulan (melihat pergeseran mix)
brand_top = brand.head(8).index
pivot = (d[d['BRAND'].isin(brand_top)]
         .pivot_table(index='BRAND', columns='BULAN', values='JUMLAH', aggfunc='sum', fill_value=0) / 1e6)
pivot.columns = pivot.columns.strftime('%b')

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlGnBu')
ax.set_xticks(range(len(pivot.columns)), pivot.columns)
ax.set_yticks(range(len(pivot.index)), pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        ax.text(j, i, f'{v:.0f}', ha='center', va='center',
                color='white' if v > pivot.values.max() * 0.6 else 'black', fontsize=9)
ax.set_title('Omzet per Brand per Bulan (juta Rp)')
ax.grid(False)
fig.colorbar(im, ax=ax, label='juta Rp')
plt.tight_layout()
plt.show()

## 6. Temuan awal

Sel di bawah merangkum angka-angka kunci secara otomatis agar tidak ada nilai yang ditulis manual.

In [ ]:
bln_max = bulanan['omzet'].idxmax()
bln_min = bulanan['omzet'].idxmin()

print('RINGKASAN TEMUAN')
print('=' * 62)
print(f"Periode                 : {d['TGL'].min():%d %b %Y} - {d['TGL'].max():%d %b %Y} ({hari_aktif} hari aktif)")
print(f"Total omzet             : Rp {d['JUMLAH'].sum():,.0f}")
print(f"Bulan tertinggi         : {bln_max:%b %Y} (Rp {bulanan.loc[bln_max, 'omzet']:,.0f})")
print(f"Bulan terendah          : {bln_min:%b %Y} (Rp {bulanan.loc[bln_min, 'omzet']:,.0f})")
print(f"Konsentrasi produk      : {n80} dari {len(produk)} SKU = 80% omzet")
print(f"Konsentrasi pelanggan   : {n80c} dari {len(pelanggan)} pelanggan = 80% omzet")
print(f"Produk teratas          : {produk.index[0][1]} (Rp {produk['omzet'].iloc[0]:,.0f})")
print(f"Pelanggan teratas       : {pelanggan.index[0]} (Rp {pelanggan['omzet'].iloc[0]:,.0f})")
print(f"Brand teratas           : {brand.index[0]} ({brand.iloc[0] / d['JUMLAH'].sum() * 100:.1f}% omzet)")
print(f"Satuan dominan          : {satuan['omzet'].idxmax()} "
      f"({satuan['omzet'].max() / d['JUMLAH'].sum() * 100:.1f}% omzet)")
print('-' * 62)
print('Catatan kualitas data (bukan penjumlahan; satu baris bisa kena >1 isu):')
for k, v in cek.items():
    if v:
        print(f'  - {k}: {v} baris')
print(f'  - kode barang dengan >1 ejaan nama: {len(nama_ganda)} kode (sudah dikanonikkan)')
print('=' * 62)

### Langkah lanjutan yang disarankan

1. **Nomor faktur** — data tidak punya ID transaksi, sehingga "order" hanya bisa didekati lewat kombinasi `TGL + NAMA`. Jika kolom faktur tersedia, analisis keranjang belanja dan rata-rata nilai order jadi jauh lebih akurat.
2. **Segmentasi RFM** pelanggan (Recency, Frequency, Monetary) memakai `TGL`, jumlah hari beli, dan omzet.
3. **Market basket** — produk apa yang sering dibeli bersamaan dalam satu kunjungan.
4. **Margin** — dataset hanya memuat harga jual; tanpa harga pokok, analisis profitabilitas belum bisa dilakukan.
5. **Efek pandemi** — periode April–September 2020 bertepatan dengan PSBB; perbandingan dengan periode 2019 akan memperjelas anomali tren.